# Kaggle Labeling — CoT-ComplianceBench

Labels `generations_<model>.jsonl` files with a **3-step resumable pipeline**:

1. **Output safety** — WildGuard on `final_answer` → `labels_output_<model>.jsonl`
2. **CoT safety** — local Qwen2.5 judge on `cot_trace` → `labels_cot_<model>.jsonl`
3. **Merge** — join + compliance quadrants (Q1–Q4) + CDR → `labeled_<model>.jsonl`

**Before running**
- Enable **GPU** (T4 ×2 recommended for the CoT judge step).
- Add your generation files as **Input Data** (Kaggle Dataset), or resume from a prior notebook commit under `/kaggle/working/cot-compliance-safety/data/raw/`.
- After the run: **Save Version → Save & Run All (Commit)** to persist `/kaggle/working`.

In [ ]:
# --- edit these before running ---
MODELS = [
    "deepseek-r1-7b",
    "deepseek-r1-8b",
    "deepseek-r1-14b",
    "qwen3-14b",
]

# WildGuard batch size (lower if you hit OOM)
OUTPUT_BATCH = 8

# CoT judge: "14b" fits 1× T4; use "32b" with JUDGE_TP=2 on 2× T4
JUDGE = "14b"
JUDGE_TP = 1

In [ ]:
# Cell 1: clone repo (explicit folder name matches src/config.py)
!git clone https://github.com/bharathgaddam1712/COT-COMPLIANCE-SAFETY.git cot-compliance-safety
%cd /kaggle/working/cot-compliance-safety

In [ ]:
# Cell 2: install dependencies
!pip -q install -r requirements.txt

In [ ]:
# Cell 3: restore generation JSONL into persistent working storage
import glob
import shutil
from pathlib import Path

RAW = Path("/kaggle/working/cot-compliance-safety/data/raw")
RAW.mkdir(parents=True, exist_ok=True)

patterns = [
    "/kaggle/input/*/data/raw/generations_*.jsonl",
    "/kaggle/input/*/generations_*.jsonl",
    "/kaggle/input/*/*.jsonl",
]
copied = []
for pattern in patterns:
    for src in glob.glob(pattern):
        name = Path(src).name
        if not name.startswith("generations_"):
            continue
        dst = RAW / name
        if not dst.exists() or dst.stat().st_size < Path(src).stat().st_size:
            shutil.copy2(src, dst)
            copied.append(name)

print("RAW_DIR:", RAW)
print("generation files:", sorted(p.name for p in RAW.glob("generations_*.jsonl")))
print("newly copied:", sorted(set(copied)))

In [ ]:
# Cell 4: verify config paths (should point at /kaggle/working/cot-compliance-safety/data/raw)
%cd /kaggle/working/cot-compliance-safety/src
!python -c "from config import BASE, RAW_DIR, JOB_LIST; print('BASE=', BASE); print('RAW_DIR=', RAW_DIR); print('JOB_LIST=', JOB_LIST)"

## Step 1 — Output safety (WildGuard)

Scores whether the **final answer** is harmful. Resumable: re-running skips IDs already in `labels_output_<model>.jsonl`.

In [ ]:
models_arg = " ".join(MODELS)
!python label_output.py --models {models_arg} --batch {OUTPUT_BATCH}

## Step 2 — CoT safety (local vLLM judge)

Scores whether the **reasoning trace itself** is unsafe, independent of the final answer. Uses Qwen2.5 AWQ in 4-bit via vLLM.

If this cell OOMs, set `JUDGE = "14b"` and `JUDGE_TP = 1`, or label one model per notebook session.

In [ ]:
models_arg = " ".join(MODELS)
!python label_cot_local.py --models {models_arg} --judge {JUDGE} --tp {JUDGE_TP}

## Step 3 — Merge labels + quadrants

Joins both label channels and writes `labeled_<model>.jsonl` with `compliance_quadrant` and `divergent` (Q3 flag).

In [ ]:
models_arg = " ".join(MODELS)
!python combine_labels.py --models {models_arg}

In [ ]:
# Cell 8: quick sanity check on outputs
from pathlib import Path
import json
from collections import Counter

RAW = Path("/kaggle/working/cot-compliance-safety/data/raw")
for model in MODELS:
    path = RAW / f"labeled_{model}.jsonl"
    if not path.exists():
        print(f"[missing] {path.name}")
        continue
    quads = Counter()
    with open(path) as f:
        for line in f:
            if line.strip():
                quads[json.loads(line).get("compliance_quadrant")] += 1
    print(f"{path.name}: {sum(quads.values())} rows  quadrants={dict(quads)}")